# Fashion-MNIST — diễn giải visual itemsets

Notebook P4 đọc các mining artifacts đã commit, dựng centroid/patch/image witnesses, retrieval và thống kê lớp nếu nhãn có thể xác minh. Không gọi lại Eclat. Phải chạy từ kernel Julia sạch sau khi P3 hoàn thành.

In [1]:
# P4 setup
using Pkg, TOML, Serialization
function find_repo_root(start)
    current = realpath(start)
    while true
        if isfile(joinpath(current, "Project.toml")) && isfile(joinpath(current, "src/EclatFIM.jl"))
            return current
        end
        parent = dirname(current)
        parent == current && error("Không tìm thấy gốc repository từ $(start)")
        current = parent
    end
end
REPO_ROOT = find_repo_root(pwd())
Pkg.activate(REPO_ROOT)
OUTPUT_DIR = joinpath(REPO_ROOT, "results/fashion/fim/B0_K32_seed123_S0")
SOURCE_DIR = joinpath(REPO_ROOT, "results/fashion/tc/tc6_configs/S0/FashionMNIST_T2_S0_B0_K32_seed123")
RAW_DIR = joinpath(REPO_ROOT, "data/raw/fashion-mnist")
PYTHON = Sys.which("python3")
isnothing(PYTHON) && error("Không tìm thấy python3")
INTERPRETATION = joinpath(REPO_ROOT, "scripts/fashion_fim/interpretation.py")
CODEBOOK_PATH = joinpath(SOURCE_DIR, "codebook.bin")

@assert isfile(joinpath(OUTPUT_DIR, "analysis/representative_patterns.csv")) "Chạy P3 trước"
@assert isfile(joinpath(SOURCE_DIR, "image_metadata.csv")) "Thiếu metadata nguồn TC6"
# Codebook, patch assignments, ảnh và labels là assets tùy chọn cho phần tương ứng.
# Nếu thiếu asset trực quan, P4 vẫn chạy retrieval chính xác và các phân tích lớp còn đủ dữ liệu.
(; output_dir=OUTPUT_DIR, source_dir=SOURCE_DIR, raw_dir=RAW_DIR, julia=string(VERSION))


  Activating project at `/run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining`


(output_dir = "/run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/results/fashion/fim/B0_K32_seed123_S0", source_dir = "/run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/results/fashion/tc/tc6_configs/S0/FashionMNIST_T2_S0_B0_K32_seed123", raw_dir = "/run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/data/raw/fashion-mnist", julia = "1.13.3")

## P4 — Xác minh nguồn và tạo centroid CSV
Codebook đã lưu giữ 16 chiều × 32 visual words. Nếu có `codebook.bin`, notebook xuất mỗi centroid thành 16 dòng với chỉ số pixel 1-based, giữ vector theo thứ tự column-major như lúc fit. Tiếp đó script xác minh mapping TID/image, itemset support, patch witnesses, ảnh Fashion-MNIST và nhãn nếu có. Thiếu codebook, patch assignments hoặc ảnh sẽ chặn gallery/witness trực quan, nhưng retrieval chính xác và phân tích lớp vẫn chạy khi còn đủ nguồn.

In [3]:
# P4 — Centroids + gallery / retrieval (không chạy Eclat)
interpretation_dir = joinpath(OUTPUT_DIR, "interpretation")
mkpath(interpretation_dir)
centroid_csv = joinpath(interpretation_dir, "codebook_centroids.csv")
if isfile(CODEBOOK_PATH)
    codebook = open(deserialize, CODEBOOK_PATH)
    @assert hasproperty(codebook, :centroids) "codebook.bin không có trường centroids"
    centroids = getproperty(codebook, :centroids)
    @assert size(centroids) == (16, 32) "Kỳ vọng ma trận centroids 16×32"
    @assert all(isfinite, centroids) "Centroid chứa giá trị không hữu hạn"
    centroid_tmp = centroid_csv * ".tmp"
    open(centroid_tmp, "w") do io
        println(io, "word_id,pixel_index,centroid_value")
        for word_id in 1:32, pixel_index in 1:16
            println(io, word_id, ",", pixel_index, ",", repr(Float64(centroids[pixel_index, word_id])))
        end
        flush(io)
    end
    mv(centroid_tmp, centroid_csv; force=true)
else
    isfile(centroid_csv) && rm(centroid_csv; force=true)
    @warn "codebook.bin is missing; continue retrieval/class analysis and report visual gallery as BLOCKED_MISSING_ASSET"
end
run(Cmd([PYTHON, INTERPRETATION, "--output-dir", OUTPUT_DIR, "--raw-dir", RAW_DIR]))


{
  "status": "PASS",
  "representative_count": 16,
  "label_join": "PASS",
  "label_provenance": "image_metadata.csv verified against raw IDX label archives",
  "class_totals": {
    "0": 7000,
    "1": 7000,
    "2": 7000,
    "3": 7000,
    "4": 7000,
    "5": 7000,
    "6": 7000,
    "7": 7000,
    "8": 7000,
    "9": 7000
  },
  "class_perspectives": {
    "common": {
      "pattern_id": "P-5f9c4ab08cac7457e9111a30e4664920607ea2c115a1433d7be98e97e64244ca",
      "items": [
        26
      ],
      "support_count": 48922,
      "class_coverage": 10
    },
    "class_enriched": {
      "pattern_id": "P-b99befb844a924231ac67eccaae468cb778c4114752965c78ae4a8859cf10d4b",
      "items": [
        14,
        23
      ],
      "class_name": "Ankle boot",
      "pattern_class_support": 1332,
      "class_total": 7000,
      "support_fraction_within_class": 0.19028571428571428,
      "support_fraction_global": 0.06882857142857143,
      "enrichment": 2.7646326276463262
    },
    "cross_c

Process(`/usr/bin/python3 /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/scripts/fashion_fim/interpretation.py --output-dir /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/results/fashion/fim/B0_K32_seed123_S0 --raw-dir /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/data/raw/fashion-mnist`, ProcessExited(0))

## Kết quả và giới hạn
Gallery và retrieval là bằng chứng máy tạo; annotations mặc định `PENDING_REVIEW`, `visual_coherence=NA`. Nếu không có nhãn hoàn chỉnh, notebook vẫn tạo witness/gallery/retrieval và ghi `BLOCKED_MISSING_LABELS`; không dựng class metrics/heatmap giả. Nhãn chỉ được dùng hậu nghiệm.

In [4]:
# P4 — Đọc và hiển thị tóm tắt interpretation đã lưu, không chạy lại mining.
summary_path = joinpath(OUTPUT_DIR, "interpretation/interpretation_summary.json")
@assert isfile(summary_path) "Interpretation chưa tạo summary"
print(read(summary_path, String))


{
  "status": "PASS",
  "representative_count": 16,
  "label_join": "PASS",
  "label_provenance": "image_metadata.csv verified against raw IDX label archives",
  "class_totals": {
    "0": 7000,
    "1": 7000,
    "2": 7000,
    "3": 7000,
    "4": 7000,
    "5": 7000,
    "6": 7000,
    "7": 7000,
    "8": 7000,
    "9": 7000
  },
  "class_perspectives": {
    "common": {
      "pattern_id": "P-5f9c4ab08cac7457e9111a30e4664920607ea2c115a1433d7be98e97e64244ca",
      "items": [
        26
      ],
      "support_count": 48922,
      "class_coverage": 10
    },
    "class_enriched": {
      "pattern_id": "P-b99befb844a924231ac67eccaae468cb778c4114752965c78ae4a8859cf10d4b",
      "items": [
        14,
        23
      ],
      "class_name": "Ankle boot",
      "pattern_class_support": 1332,
      "class_total": 7000,
      "support_fraction_within_class": 0.19028571428571428,
      "support_fraction_global": 0.06882857142857143,
      "enrichment": 2.7646326276463262
    },
    "cross_c

## P5 handoff
1. Mở `interpretation/pattern_gallery/index.html` để review evidence thật.
2. Reviewer cập nhật `pattern_annotations.csv`; không điền coherence trước khi xem card.
3. Quay lại notebook mining để chạy report/audit sau khi interpretation hoàn tất.
4. Ghi MCP log chỉ từ run/restart/save/reopen đã quan sát thực tế. Notebook này không chạy Eclat và không ghi nhãn APPROVED tự động.